In [16]:
import pandas as pd

file_path = "All_Brokers_All_Emails_Combined.xlsx"
sheets = pd.read_excel(file_path, sheet_name=None)

import re

def process_vessel_name(name):
    if not isinstance(name, str):
        return None, False, False
    
    original = name.strip()
    s = original.upper()
    
    # Flags
    is_oos = bool(re.search(r"\bOOS\b", s))
    is_optional = bool(re.search(r"\bOR\b", s))
    
    # Remove parentheses
    clean = re.sub(r"\(.*?\)", "", original)
    
    # Remove PMAX / OS / OOS text but DO NOT remove "or"
    clean = re.sub(r"\bPMAX\b", "", clean, flags=re.IGNORECASE)
    clean = re.sub(r"\bOS\b", "", clean, flags=re.IGNORECASE)
    clean = re.sub(r"\bOOS\b", "", clean, flags=re.IGNORECASE)
    
    clean = re.sub(r"\s{2,}", " ", clean).strip()
    
    return clean.title(), is_oos, is_optional


standard_rows = []

for name, df in sheets.items():

    # --------------------------
    # BRS FIXTURES (USG only)
    # --------------------------
    if "BRS_FIXTURES" in name:

        df_usg = df[df["Route"] == "USG"]

        for _, r in df_usg.iterrows():
            standard_rows.append({
                "Broker": r.get("Broker"),
                "Vessel": r.get("Vessel"),
                "Counterparty": r.get("Charterer"),
                "Route": r.get("Route"),
                "ETA Start": r.get("Laycan Start"),
                "ETA End": r.get("Laycan End"),
                "ETA Midpoint": r.get("Laycan Midpoint"),
                "Notes": r.get("Raw Line"),
                "Email Sent Date": r.get("Email Sent Date")
            })

    # --------------------------
    # Fearnleys
    # --------------------------
    elif "Fearnleys" in name:

        for _, r in df.iterrows():
            standard_rows.append({
                "Broker": r.get("Broker"),
                "Vessel": r.get("Vessel"),
                "Counterparty": r.get("Owner"),
                "Route": "USG",
                "ETA Start": r.get("ETA Start"),
                "ETA End": r.get("ETA End"),
                "ETA Midpoint": r.get("ETA Midpoint"),
                "Notes": r.get("Notes"),
                "Email Sent Date": r.get("Email Sent Date")
            })

    # --------------------------
    # Poten
    # --------------------------
    elif "Poten" in name:

        for _, r in df.iterrows():
            standard_rows.append({
                "Broker": r.get("Broker"),
                "Vessel": r.get("Vessel"),
                "Counterparty": r.get("Owner"),
                "Route": "USG",
                "ETA Start": r.get("ETA USG Start"),
                "ETA End": r.get("ETA USG End"),
                "ETA Midpoint": r.get("ETA USG Midpoint"),
                "Notes": r.get("Additional Comments"),
                "Email Sent Date": r.get("Email Sent Date")
            })

    # --------------------------
    # Gibson
    # --------------------------
    elif "Gibson" in name:

        for _, r in df.iterrows():
            standard_rows.append({
                "Broker": r.get("Broker"),
                "Vessel": r.get("Vessel"),
                "Counterparty": r.get("OWNER"),
                "Route": "USG",
                "ETA Start": r.get("ETA USG Start"),
                "ETA End": r.get("ETA USG End"),
                "ETA Midpoint": r.get("ETA USG Midpoint"),
                "Notes": r.get("COMMENTS"),
                "Email Sent Date": r.get("Email Sent Date")
            })

    # --------------------------
    # Affinity
    # --------------------------
    elif "Affinity" in name:

        for _, r in df.iterrows():
            standard_rows.append({
                "Broker": r.get("Broker"),
                "Vessel": r.get("Vessel"),
                "Counterparty": r.get("Control"),
                "Route": "USG",
                "ETA Start": r.get("ETA Start"),
                "ETA End": r.get("ETA End"),
                "ETA Midpoint": r.get("ETA Midpoint"),
                "Notes": r.get("Notes"),
                "Email Sent Date": r.get("Email Sent Date")
            })

# Build unified dataframe
standard_df = pd.DataFrame(standard_rows)

# Sort chronologically
standard_df = standard_df.sort_values("ETA Midpoint")

standard_df[["Vessel_Clean", "Is_OOS", "Is_Optional"]] = (
    standard_df["Vessel"]
    .apply(lambda x: pd.Series(process_vessel_name(x)))
)

print(standard_df.head())

       Broker                    Vessel Counterparty Route  ETA Start  \
52     Gibson                 BW BREEZE       BW LPG   USG 2026-03-14   
37  Fearnleys  BW Breeze or BW Magellan           BW   USG 2026-03-14   
67      Poten            BW Magellan os           BW   USG 2026-03-16   
0    Affinity                    BW TBN           BW   USG 2026-03-14   
17   Affinity                    BW TBN           BW   USG 2026-03-14   

      ETA End        ETA Midpoint  \
52 2026-03-15 2026-03-14 12:00:00   
37 2026-03-17 2026-03-15 12:00:00   
67 2026-03-17 2026-03-16 12:00:00   
0  2026-03-20 2026-03-17 00:00:00   
17 2026-03-20 2026-03-17 00:00:00   

                                          Notes      Email Sent Date  \
52  VIA CAPE. OOS MAGELLAN 16-17 MAR VIA PANAMA  2026-02-27 16:01:30   
37                   via Cape / via Pan NB 12th  2026-02-27 16:01:35   
67                   Northbound Panama 12 March  2026-02-27 16:01:16   
0                                    via Panama  2

In [17]:
standard_df.to_excel("Single_Table_Combined.xlsx", index=False)